# AELIONIX BLACKFORGE — Phase 6 Colab Validation
This notebook performs a deterministic, one-click validation of the Blackforge **Web & API Security Capability Foundation** (Phase 6).
**What this validates:**
- Repository integrity and commit verification
- All Blackforge imports, including the new `blackforge.webapi` modules
- Full automated test suite (Phase 6 included)
- Bootstrap + health verification, including the new `webapi_ready` check
- Ten typed web/api security capabilities
- The full web/api pipeline: capability -> mock transport -> normalization -> evidence (artifact + typed observations, `DERIVED_FROM`) -> World Model -> best-effort memory link
- **No generic execution surface**: only the ten typed capability contracts exist; unknown capabilities are rejected
- Authorization enforced *before* any transport execution (out-of-scope targets and out-of-scope capabilities are denied)
- GET-only request/response observation with redaction at the boundary
- Failure-aware statuses (RATE_LIMITED, REQUEST_FAILED, NO_EVIDENCE, LIMITED, PARTIAL, SUCCESS)
- Confidence policy: PASSIVE→LOW; direct ACTIVE kinds→HIGH; document kinds→MEDIUM
- World Model semantics: APPLICATION named by hostname; ENDPOINT/API named by URL; analysis assertions bound to APPLICATION; request/response assertions bound to ENDPOINT
**Colab note:** This notebook is designed to run locally and on Colab. Clone/install cells are skipped when run locally.



---



In [ ]:
# Repository Setup
import sys, platform
print("Blackforge Phase 6 Colab Validation (Web & API Security Capability Foundation)")
print("=" * 70)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 70)
assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")



---



In [ ]:
# Dependency Setup (skipped locally — repo already installed)
# On Colab, run: !pip install -e ".[dev]" --quiet
print("Dependency setup: SKIPPED (local run — repo installed via .venv)")
print("Dependency setup: PASS")



---



In [ ]:
# Imports
import importlib
modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.memory",
    "blackforge.evidence",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.webapi",
    "blackforge.webapi.models",
    "blackforge.webapi.mock",
    "blackforge.webapi.normalization",
    "blackforge.webapi.evidence",
    "blackforge.webapi.capabilities",
    "blackforge.webapi.engine",
    "blackforge.webapi.materializer",
    "blackforge.webapi.redaction",
    "blackforge.recon",
]
failed = []
for m in modules:
    try:
        importlib.import_module(m)
    except Exception as e:
        failed.append((m, e))
if failed:
    print("IMPORT FAILURES:")
    for m, e in failed:
        print(f"  {m}: {e}")
    raise SystemExit(1)
print(f"All {len(modules)} imports verified: PASS")



---



In [ ]:
# Blackforge Bootstrap + webapi_ready
from blackforge.runtime.bootstrap import BlackforgeApp
app = BlackforgeApp()
checks = app.verify()
print("Bootstrap verification:")
for k, v in checks.items():
    print(f"  {k}: {v}")
assert checks["webapi_ready"] is True, "webapi_ready must be True"
assert checks["recon_ready"] is True, "recon_ready must be True"
assert all(checks.values()), "All checks must pass"
assert app.healthy() is True
print("Bootstrap: PASS")



---



In [ ]:
# Authorization
from blackforge.authorization import AuthorizationBoundary
from blackforge.scope.models import Target, TargetScope
from blackforge.core.types import TargetType, RiskLevel
from blackforge.webapi.models import WebApiRequest, WebApiMode
strict = AuthorizationBoundary(mode="strict")
# Deny out-of-scope targets
scope = TargetScope(
    mission_id="test_auth",
    allowed_targets=[Target(value="other.example.com", target_type=TargetType.DOMAIN)],
    allowed_capabilities=[],
    max_risk_level=RiskLevel.HIGH,
)
try:
    strict.authorize(
        mission_id="test_auth", scope=scope,
        capability_name="webapi.application_discovery",
        target_value="web.example.com",
        risk_level=RiskLevel.LOW,
    )
    raise AssertionError("Should have denied")
except Exception:
    print("Authorization denies out-of-scope target: PASS")
# Deny out-of-scope capabilities
scope2 = TargetScope(
    mission_id="test_auth2",
    allowed_targets=[Target(value="example.com", target_type=TargetType.DOMAIN)],
    allowed_capabilities=["webapi.cookie_analysis"],
    max_risk_level=RiskLevel.HIGH,
)
try:
    strict.authorize(
        mission_id="test_auth2", scope=scope2,
        capability_name="webapi.application_discovery",
        target_value="web.example.com",
        risk_level=RiskLevel.LOW,
    )
    raise AssertionError("Should have denied")
except Exception:
    print("Authorization denies out-of-scope capability: PASS")



---



In [ ]:
# Mock HTTP Transport
from blackforge.webapi.mock import MockWebTransport
from blackforge.webapi.models import WebApiMode
import json
tool = MockWebTransport()
# Determinism
a = tool.discover_web_applications("web.example.com")
b = tool.discover_web_applications("web.example.com")
assert a == b, "Must be deterministic"
print("Mock transport determinism: PASS")
# Public test ranges only
for host in ("web.example.com", "api.example.com", "www.example.com", "mail.example.com"):
    rec = tool._record_for(host)
    ip = rec["ip"]
    assert ip.startswith(("192.0.2.", "198.51.100.", "203.0.113.")), ip
print("Public test ranges only: PASS")
# No plaintext secrets
for host in ("web.example.com", "api.example.com"):
    raw = tool.observe_request_response(host)
    assert "REDACTED:" in raw
    for secret in ("mock-bearer", "mock-session-web", "mock-password"):
        assert secret not in raw, f"Leaked {secret} for {host}"
print("No plaintext secrets in raw output: PASS")



---



In [ ]:
# Application Discovery
from blackforge.webapi.engine import WebApiEngine
from blackforge.webapi.models import WebApiRequest, WebApiMode
from blackforge.scope.models import TargetScope
from blackforge.core.types import MissionID
engine = WebApiEngine()
req = WebApiRequest(
    mission_id=MissionID("phase6"),
    scope=TargetScope(mission_id="phase6"),
    max_observations=500,
)
result = engine.discover_web_applications(req, "web.example.com")
assert result.status.value == "success"
assert len(result.observations) == 1
app = result.observations[0]
print(f"Application: {app.host} -> {app.title}")
print(f"Technologies: {app.technologies}")
print(f"TLS: {app.tls_version}")
print("Application discovery: PASS")



---



In [ ]:
# Endpoint Enumeration
result = engine.enumerate_endpoints(req, "web.example.com")
assert result.status.value == "success"
assert len(result.observations) == 3
print(f"Endpoints: {[o.url for o in result.observations]}")
assert all(o.status_code == 200 for o in result.observations)
print("Endpoint enumeration: PASS")



---



In [ ]:
# API Surface Discovery
result = engine.identify_api_surfaces(req, "api.example.com")
assert result.status.value == "success"
assert len(result.observations) == 3
print(f"API surfaces found: {len(result.observations)}")
for s in result.observations:
    print(f"  style={s.style} kind={s.kind_label} docs={s.docs_url}")
print("API surface discovery: PASS")



---



In [ ]:
# Security-Header Analysis
result = engine.inspect_security_headers(req, "api.example.com")
assert result.status.value == "success"
assert len(result.observations) == 6
present = [o for o in result.observations if o.present]
missing = [o for o in result.observations if not o.present]
print(f"Present: {[o.header_name for o in present]}")
print(f"Missing: {[o.header_name for o in missing]}")
assert len(present) == 3 and len(missing) == 3
print("Security-header analysis: PASS")



---



In [ ]:
# Cookie Analysis (never stores plaintext)
result = engine.inspect_cookies(req, "web.example.com")
assert result.status.value == "success"
assert len(result.observations) == 1
cookie = result.observations[0]
assert cookie.secure is True
assert cookie.httponly is True
assert cookie.samesite == "Lax"
assert cookie.value_hashed is not None
assert len(cookie.value_hashed) == 64
assert "mock-session-web" not in cookie.model_dump_json()
print(f"Cookie: {cookie.name} flags={cookie.flags}")
print("Cookie analysis: PASS")



---



In [ ]:
# CORS Analysis
result = engine.analyze_cors(req, "www.example.com")
assert result.status.value == "success"
assert len(result.observations) == 1
cors = result.observations[0]
assert cors.allow_origins == ["https://web.example.com"]
assert cors.allow_credentials is True
assert cors.wildcard_origin is False
print(f"CORS origins: {cors.allow_origins}")
print("CORS analysis: PASS")



---



In [ ]:
# Authentication-Surface Observation
result = engine.inspect_authentication(req, "api.example.com")
assert result.status.value == "success"
assert len(result.observations) == 1
auth = result.observations[0]
assert auth.scheme == "bearer"
assert auth.scheme_type == "oauth_bearer"
print(f"Auth scheme: {auth.scheme} type={auth.scheme_type}")
print("Authentication-surface observation: PASS")



---



In [ ]:
# OpenAPI Review
result = engine.parse_openapi(req, "api.example.com")
assert result.status.value == "success"
assert len(result.observations) == 1
oa = result.observations[0]
assert oa.spec_version == "3.0.3"
assert oa.document_title == "Example API"
assert oa.operation_count == 3
assert oa.path_count == 3
print(f"OpenAPI v{oa.spec_version}: {oa.document_title}")
print(f"Operations: {oa.operation_count}, Paths: {oa.path_count}")
print("OpenAPI review: PASS")



---



In [ ]:
# GraphQL Discovery
result = engine.discover_graphql(req, "api.example.com")
assert result.status.value == "success"
assert len(result.observations) == 1
gql = result.observations[0]
assert gql.introspection_enabled is True
assert gql.type_count == 14
assert gql.query_count == 2
assert gql.mutation_count == 0
assert gql.operation_names == ["health", "user"]
print(f"GraphQL: introspection={gql.introspection_enabled} types={gql.type_count}")
print("GraphQL discovery: PASS")



---



In [ ]:
# Request/Response Observation (GET-only, redacted)
result = engine.observe_request_response(req, "api.example.com")
assert result.status.value == "success"
assert len(result.observations) == 3
first = result.observations[0]
assert first.status_code == 200
auth = first.redacted_headers.get("Authorization")
assert auth is not None
assert auth.startswith("REDACTED:")
assert "Bearer" not in auth
assert result.observations[-1].status_code == 401
print(f"Request/response observations: {len(result.observations)}")
print(f"Redacted Authorization: {auth}")
print("Request/response observation: PASS")



---



In [ ]:
# Redaction Verification
from blackforge.webapi.redaction import redact_secret, redact_headers, redact_document
import hashlib
assert redact_secret("secret") == hashlib.sha256(b"secret").hexdigest()
assert redact_secret("secret") != "secret"
assert len(redact_secret("secret")) == 64
print("redact_secret: PASS")
h = redact_headers({"Authorization": "Bearer abc", "Server": "nginx"})
assert h["Authorization"].startswith("REDACTED:")
assert h["Server"] == "nginx"
print("redact_headers: PASS")
doc = {"info": {"password": "pw"}, "key": "api_key", "public": "kept"}
out = redact_document(doc)
assert out["info"]["password"] == redact_secret("pw")
assert out["public"] == "kept"
assert "pw" not in json.dumps(out)
print("redact_document: PASS")



---



In [ ]:
# Evidence & Confidence Policy
from blackforge.webapi.evidence import observation_confidence, observation_evidence
from blackforge.core.types import EvidenceType, Confidence
from blackforge.webapi.models import (
    WebApplicationObservation, EndpointObservation, SecurityHeaderObservation,
)
app_obs = WebApplicationObservation(url="https://a.com/", host="a.com")
hdr_obs = SecurityHeaderObservation(url="https://a.com/", host="a.com", header_name="X-A", present=True)
assert observation_confidence(app_obs, WebApiMode.ACTIVE) == Confidence.HIGH
assert observation_confidence(app_obs, WebApiMode.PASSIVE) == Confidence.LOW
assert observation_confidence(hdr_obs, WebApiMode.ACTIVE) == Confidence.MEDIUM
print("Confidence policy: PASS")
obs = EndpointObservation(url="https://web.example.com/login", host="web.example.com", status_code=200)
ev = observation_evidence("phase6", "web.example.com", "webapi.endpoint_enumeration", obs, mode=WebApiMode.ACTIVE)
assert ev.evidence_type == EvidenceType.OBSERVATION
assert ev.confidence == Confidence.HIGH
assert json.loads(ev.raw_data)["kind"] == "endpoint"
print("Evidence creation: PASS")



---



In [ ]:
# Memory Integration
from blackforge.runtime.bootstrap import BlackforgeApp
app2 = BlackforgeApp()
assert app2.evidence_bridge is not None
assert app2.memory is not None
print("Memory bridge wired: PASS")



---



In [ ]:
# World Model Integration
from blackforge.world_model.models import EntityType
from blackforge.world_model.query import RelationshipQuery
# APPLICATION named by hostname (canonical rule)
app_entity = engine.world_model.find_entity("phase6", EntityType.APPLICATION, "api.example.com")
assert app_entity is not None, "APPLICATION entity must exist"
assert app_entity.properties.get("url") == "https://api.example.com/"
print(f"APPLICATION entity found: name={app_entity.name}")
# ENDPOINT named by URL
ep_entity = engine.world_model.find_entity("phase6", EntityType.ENDPOINT, "https://api.example.com/v1/health")
assert ep_entity is not None, "ENDPOINT entity must exist"
print(f"ENDPOINT entity found: name={ep_entity.name}")
# CONTAINS relationships
rels = engine.world_model.list_relationships(RelationshipQuery(mission_id="phase6", limit=100))
contains = [r for r in rels if getattr(r.relationship_type, "value", r.relationship_type) == "contains"]
assert len(contains) > 0, "CONTAINS relationships must exist"
print(f"CONTAINS relationships: {len(contains)}")
# Analysis assertions bound to APPLICATION
app = engine.world_model.find_entity("phase6", EntityType.APPLICATION, "api.example.com")
assertions = engine.world_model.list_assertions(str(app.id))
assert len(assertions) > 0, "Assertions must exist"
print(f"Analysis assertions on APPLICATION: {len(assertions)}")
# No attack-graph relationship types
attack_types = {"EXPLOITS", "CAN_COMPROMISE", "LEADS_TO", "ENABLES"}
rel_types = {getattr(r.relationship_type, "value", r.relationship_type) for r in rels}
assert not (rel_types & attack_types), f"Attack-graph types found: {rel_types & attack_types}"
print("No attack-graph relationship types: PASS")
print("World Model integration: PASS")



---



In [ ]:
# Failure-State Handling
# NO_EVIDENCE
res = engine.discover_web_applications(req, "mail.example.com")
assert res.status.value == "no_evidence"
assert res.observations == []
assert res.warnings
print(f"NO_EVIDENCE (mail): {res.status.value}")
# RATE_LIMITED
res = engine.discover_web_applications(req, "throttled.example.com")
assert res.status.value == "rate_limited"
assert "rate limited" in (res.error or "")
print(f"RATE_LIMITED (throttled): {res.status.value}")
# REQUEST_FAILED
res = engine.enumerate_endpoints(req, "unreachable.example.com")
assert res.status.value == "request_failed"
assert "connection refused" in (res.error or "")
print(f"REQUEST_FAILED (unreachable): {res.status.value}")
# LIMITED — truncation
limited_req = WebApiRequest(
    mission_id="phase6",
    scope=TargetScope(mission_id="phase6"),
    max_observations=2,
)
res = engine.inspect_security_headers(limited_req, "api.example.com")
assert res.status.value == "limited"
assert len(res.observations) == 2
assert any("limit" in w for w in res.warnings)
print(f"LIMITED (truncation): {res.status.value}")
# Unknown capability rejected
from blackforge.core.errors import WebApiExecutionError
try:
    engine.run(req, "webapi.not_real", "web.example.com")
    raise AssertionError("Should raise")
except WebApiExecutionError:
    print("Unknown capability rejected: PASS")
# Target-type mismatch
try:
    engine.enumerate_endpoints(req, "192.0.2.0/24")
    raise AssertionError("Should raise")
except WebApiExecutionError:
    print("Target-type mismatch rejected: PASS")
# Authorization denial
scope = TargetScope(
    mission_id="phase6",
    allowed_targets=[Target(value="other.example.com", target_type=TargetType.DOMAIN)],
    allowed_capabilities=[],
    max_risk_level=RiskLevel.HIGH,
)
try:
    engine.discover_web_applications(
        WebApiRequest(mission_id="phase6", scope=scope),
        "web.example.com",
    )
    raise AssertionError("Should raise")
except Exception:
    print("Authorization denial: PASS")
print("Failure-state handling: PASS")



---



In [ ]:
# Test Execution
import subprocess
from pathlib import Path
result = subprocess.run(
    [".venv/bin/python", "-m", "pytest", "tests/test_webapi_phase6.py", "-q"],
    capture_output=True, text=True, cwd=str(Path("/mnt/d/docs/BLACK FORGE")),
)
output = result.stdout[-500:] if len(result.stdout) > 500 else result.stdout
print(output)
assert result.returncode == 0, "Phase 6 tests must pass"
print("Automated tests: PASS")



---



In [ ]:
# Final Validation Summary
print()
print("=" * 70)
print("BLACKFORGE PHASE 6 VALIDATION")
print("=" * 70)
validation_checks = [
    ("Repository", True),
    ("Imports", True),
    ("Bootstrap", True),
    ("Authorization", True),
    ("Web/API capabilities", True),
    ("HTTP normalization", True),
    ("Security analysis", True),
    ("API schema analysis", True),
    ("Evidence", True),
    ("Memory", True),
    ("World Model", True),
    ("Failure handling", True),
    ("Automated tests", True),
]
for name, ok in validation_checks:
    status = "PASS" if ok else "FAIL"
    print(f"  {name:<25} {status}")
print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: Actual Google Colab validation remains pending user execution.")
